In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

In [ ]:
%cd capstone_project_GroupA
!git checkout main

In [ ]:
%cd src

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/capstone_project_GroupA/patchtst_results'

Run from here if not using Colab.

**Do not use specific_output_dir**

In [ ]:
from ModelFiles.GroupAModels import TransformersModel
from ModelFiles.ModelConfigs import TransformersConfig, SEEDS
from ModelFiles.ModelEnums import TransformerModelType
from ModelFiles.ModelPlots import *

USE_LOG_TARGET = True
BEST_MODEL_PARAMS = [(48, 720), (336, 720), (720, 336)]
for horizon, context_length in BEST_MODEL_PARAMS:
    for seed in SEEDS:
        if seed == SEEDS[0]: # Only save results for the first seed to save space
            save_prediction_results = True
        else:
            save_prediction_results = False
        patchtst_config = TransformersConfig(
            task_id=f"patchtst_best_model_h{horizon}_seed{seed}",
            model=TransformerModelType.PATCHTST,
            forecast_horizon=horizon,
            lookback_window=context_length,
            used_log_target=USE_LOG_TARGET,
            target_col= "LOG_TOTALDEMAND" if USE_LOG_TARGET else "TOTALDEMAND",
            feature_cols=['TEMPERATURE', 'TEMP_SQUARED', 'IS_WEEKEND', 'demand_1_year_ago'],
            scale=True,
            date_col='DATETIME',
            variate='MS',
            patch_len=16,
            stride=8,
            d_model=128,
            num_attention_heads=16,
            num_encoder_layers=3,
            dim_ff=256,
            dropout=0.0,
            dropout_head_fc=0.0,
            use_gpu=True,
            time_encoding='timeF',
            training_epochs=100,
            batch_size=32,
            learning_rate=0.000005,
            output_attention=False,
            lradj='TST',
            patience=10,
            seed=seed,
            eval_step_size=48,
            save_test_results=save_prediction_results,
            debug=False,
            save_training_log=False,
            save_model=False
        )
        patch_tst_model = TransformersModel(patchtst_config, specific_output_dir=SAVE_PATH)
        patch_tst_model.train_model()
        patch_tst_model.evaluate_model(test_mode=1)
        print("=" * 200)
        print("\n")